# Held-out Targeting and Policy Evaluation

This notebook evaluates whether a personalised treatment policy creates incremental profit beyond the strongest fixed policy. It uses pre-treatment features only, a treatment-stratified 70/30 train-holdout split, separate outcome models for each randomized arm, and inverse-propensity plus doubly robust policy-value estimates.

The reference commercial assumptions are a 40% contribution margin and a contact cost of 0.10 per emailed customer. These assumptions are varied in the sensitivity analysis.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_validation import load_data
from src.policy_targeting import (
    bootstrap_policy_uplift, capacity_curve, feature_importance,
    fit_t_learner, model_diagnostics, policy_sensitivity, policy_summary
)

In [ ]:
frame = load_data(PROJECT_ROOT / 'data/raw/hillstrom.csv')
result = fit_t_learner(frame, test_size=0.30, random_state=42)
summary, policies = policy_summary(result, contribution_margin=0.40, contact_cost=0.10)
summary

In [ ]:
bootstrap_policy_uplift(
    result, policies, contribution_margin=0.40, contact_cost=0.10,
    iterations=500, random_state=42
)

In [ ]:
capacity_curve(result, contribution_margin=0.40, contact_cost=0.10)

In [ ]:
policy_sensitivity(result)

In [ ]:
model_diagnostics(result), feature_importance(result).head(15)

## Interpretation rule

A personalised policy is recommended only when its held-out profit uplift over Men's-send-to-all is positive and its paired bootstrap confidence interval provides adequate evidence that the improvement is not sampling noise. Predictive model accuracy alone is not treated as proof of policy value.